# Train the floor-marker reader (YOLO)

Trains a small Ultralytics YOLO model to find the red docking frame and read which station's QR it is, also on worn, dirty and covered markers.

**Steps**
1. Runtime > Change runtime type > **T4 GPU**.
2. Run the cells in order. The upload cell asks for `markers_v1.zip` (made on the laptop by `tools/ai/marker_dataset/pack.sh`).
3. The last cell downloads `marker_reader.zip` (the trained model: `best.pt` and `best.onnx`, plus the training plots). Put it in the workspace's `ai_data/` folder and tell Claude.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.3.40 onnx onnxslim

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose markers_v1.zip
!rm -rf /content/markers_v1 && unzip -q markers_v1.zip -d /content/ && ls /content/markers_v1

In [ ]:
# Paths in data.yaml are made absolute for Colab.
import yaml
cfg = yaml.safe_load(open('/content/markers_v1/data.yaml'))
cfg['path'] = '/content/markers_v1'
yaml.safe_dump(cfg, open('/content/markers_v1/data_colab.yaml', 'w'))
cfg

In [ ]:
import numpy as np
np.trapz = np.trapezoid   # numpy 2 renamed it; Ultralytics 8.3.40 still calls the old name
from ultralytics import YOLO
# Nano model: it has to run several times a second on the laptop's CPU.
# No left-right or up-down flips: a mirrored QR is not a valid QR. Rotation is
# already covered by the dataset (every heading), colour/brightness are varied.
model = YOLO('yolo11n.pt')
model.train(data='/content/markers_v1/data_colab.yaml', imgsz=640, epochs=120, batch=32,
            patience=30, fliplr=0.0, flipud=0.0, degrees=0.0, mosaic=0.5,
            hsv_h=0.01, hsv_s=0.4, hsv_v=0.5, project='/content/runs', name='markers',
            seed=11, deterministic=True)

In [ ]:
best = YOLO('/content/runs/markers/weights/best.pt')
metrics = best.val(data='/content/markers_v1/data_colab.yaml', imgsz=640, conf=0.001)
print('mAP50-95', metrics.box.map, 'mAP50', metrics.box.map50)
best.export(format='onnx', imgsz=640, opset=12, simplify=True)

In [ ]:
!cd /content/runs/markers && zip -q -r /content/marker_reader.zip weights/best.pt weights/best.onnx results.csv results.png confusion_matrix.png PR_curve.png args.yaml
from google.colab import files
files.download('/content/marker_reader.zip')